In [ ]:
import time

from binance import ThreadedWebsocketManager

api_key = '<api_key>'
api_secret = '<api_secret>'

def main():

    symbol = 'BNBBTC'

    twm = ThreadedWebsocketManager(api_key=api_key, api_secret=api_secret)
    # start is required to initialise its internal loop
    twm.start()

    def handle_socket_message(msg):
        print(f"message type: {msg['e']}")
        print(msg)

    twm.start_kline_socket(callback=handle_socket_message, symbol=symbol)

    # multiple sockets can be started
    twm.start_depth_socket(callback=handle_socket_message, symbol=symbol)

    # or a multiplex socket can be started like this
    # see Binance docs for stream names
    streams = ['bnbbtc@miniTicker', 'bnbbtc@bookTicker']
    twm.start_multiplex_socket(callback=handle_socket_message, streams=streams)

    twm.join()


if __name__ == "__main__":
   main()

Exception in thread Thread-3:
Traceback (most recent call last):
  File "c:\Users\ASUS\anaconda3\envs\futurx\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\ASUS\anaconda3\envs\futurx\lib\site-packages\binance\ws\threaded_stream.py", line 65, in run
    self._loop.run_until_complete(self.socket_listener())
  File "c:\Users\ASUS\anaconda3\envs\futurx\lib\asyncio\base_events.py", line 625, in run_until_complete
    self._check_running()
  File "c:\Users\ASUS\anaconda3\envs\futurx\lib\asyncio\base_events.py", line 584, in _check_running
    raise RuntimeError('This event loop is already running')
RuntimeError: This event loop is already running
c:\Users\ASUS\anaconda3\envs\futurx\lib\threading.py:1018: RuntimeWarning: coroutine 'ThreadedApiManager.socket_listener' was never awaited
  self._invoke_excepthook(self)


In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Binance数据订阅测试脚本
简单演示如何订阅Binance交易所的分钟K线数据
"""

import time
import sys
import os
from datetime import datetime
from typing import List, Dict, Any
import threading

# 添加项目根目录到系统路径
__file__ = r"F:\workspace\futurx\futurx\script\crypto\debug.ipynb"
root_path = os.path.abspath(os.path.join(os.path.dirname(__file__), '../../'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 导入binance库
from binance.client import Client
from binance.exceptions import BinanceAPIException

# 导入自定义组件
from core.engine.event_engine import EventEngine
from core.engine.main_engine import MainEngine, BaseEngine
from core.constant import Event, EventType, Exchange, Interval
from core.base import BarData

class BinanceDataEngine(BaseEngine):
    """币安数据引擎"""
    
    def __init__(
        self,
        main_engine: MainEngine,
        event_engine: EventEngine,
        engine_name: str = "binance_data",
        api_key: str = "", 
        api_secret: str = ""
    ):
        """初始化"""
        super().__init__(main_engine, event_engine, engine_name)
        
        # API凭证
        self.api_key = api_key
        self.api_secret = api_secret
        
        # 初始化Binance客户端
        self.client = None
        
        # 轮询线程
        self.polling_thread = None
        self.polling_interval = 60  # 默认60秒轮询一次
        self.active = False
        
        # 订阅的交易对
        self.subscriptions = {}
        
        # 初始化连接
        self._init_client()
        
    def _init_client(self):
        """初始化Binance客户端"""
        try:
            # 使用API密钥或公共API
            self.client = Client(self.api_key, self.api_secret)
            print("Binance客户端初始化成功")
        except Exception as e:
            print(f"Binance客户端初始化失败: {e}")
            raise
    
    def _poll_kline_data(self):
        """轮询K线数据"""
        while self.active:
            for symbol in self.subscriptions:
                try:
                    # 获取最近的K线数据
                    klines = self.client.get_klines(
                        symbol=symbol.upper(),
                        interval=self.subscriptions[symbol]['interval'],
                        limit=1
                    )
                    
                    if klines and len(klines) > 0:
                        # Binance K线数据格式
                        # [开盘时间, 开盘价, 最高价, 最低价, 收盘价, 成交量, 收盘时间, 成交额, 成交笔数, 主动买入成交量, 主动买入成交额, 忽略]
                        k = klines[0]
                        
                        # 只处理已完成的K线
                        if int(k[0]) + (60 * 1000) <= int(time.time() * 1000):  # 如果K线已完成
                            # 创建BarData对象
                            bar = BarData(
                                symbol=symbol.upper(),
                                exchange=Exchange.LOCAL,
                                datetime=datetime.fromtimestamp(k[6] / 1000),
                                interval=Interval.MINUTE,
                                volume=float(k[5]),
                                turnover=float(k[7]),
                                open_interest=0,
                                open_price=float(k[1]),
                                high_price=float(k[2]),
                                low_price=float(k[3]),
                                close_price=float(k[4])
                            )
                            
                            # 推送事件
                            event = Event(
                                type=EventType.BAR1MIN,
                                data=bar
                            )
                            self.event_engine.put(event)
                            print(f"收到{symbol.upper()}的1分钟K线: 时间={bar.datetime}, 收盘价={bar.close_price}")
                            
                except BinanceAPIException as e:
                    print(f"获取{symbol}的K线数据失败: {e}")
                except Exception as e:
                    print(f"处理{symbol}的K线数据时发生错误: {e}")
                    
            # 休眠一段时间
            time.sleep(self.polling_interval)
    
    def subscribe(self, symbol: str, interval: str = '1m'):
        """订阅K线数据"""
        try:
            # 转换为Binance格式的交易对
            binance_symbol = symbol.lower()
            
            # 检查是否已订阅
            if binance_symbol in self.subscriptions:
                print(f"{symbol}已经订阅")
                return True
            
            # 记录订阅
            self.subscriptions[binance_symbol] = {
                'interval': interval
            }
            
            # 如果轮询线程未启动，则启动
            if not self.active:
                self.active = True
                self.polling_thread = threading.Thread(target=self._poll_kline_data)
                self.polling_thread.daemon = True
                self.polling_thread.start()
                
            print(f"成功订阅{symbol}的{interval}K线数据")
            return True
            
        except Exception as e:
            print(f"订阅{symbol}的{interval}K线数据失败: {e}")
            return False
    
    def unsubscribe(self, symbol: str):
        """取消订阅"""
        try:
            binance_symbol = symbol.lower()
            
            if binance_symbol not in self.subscriptions:
                print(f"{symbol}未订阅")
                return True
            
            # 删除订阅记录
            del self.subscriptions[binance_symbol]
            
            # 如果没有订阅，停止轮询线程
            if not self.subscriptions and self.active:
                self.active = False
                if self.polling_thread and self.polling_thread.is_alive():
                    self.polling_thread.join(timeout=2.0)
            
            print(f"成功取消订阅{symbol}的K线数据")
            return True
            
        except Exception as e:
            print(f"取消订阅{symbol}的K线数据失败: {e}")
            return False
    
    def stop(self):
        """停止引擎"""
        # 停止轮询线程
        self.active = False
        if self.polling_thread and self.polling_thread.is_alive():
            self.polling_thread.join(timeout=2.0)
        
        # 清空订阅
        self.subscriptions.clear()
        super().stop()
        print("Binance数据引擎已停止")


def handle_bar_data(event):
    """处理K线数据的回调函数"""
    bar = event.data
    print(f"[回调] 收到{bar.symbol}的K线数据: 开={bar.open_price}, 高={bar.high_price}, 低={bar.low_price}, 收={bar.close_price}")


def main():
    """主函数"""
    print("开始测试Binance数据订阅...")
    
    # 创建事件引擎和主引擎
    event_engine = EventEngine()
    main_engine = MainEngine()
    
    # 注册事件处理函数
    event_engine.register(EventType.BAR1MIN, handle_bar_data)
    
    # 启动事件引擎
    event_engine.start()
    print("事件引擎已启动")
    
    # 创建Binance数据引擎
    # 如需使用API密钥，在这里传入
    binance_engine = BinanceDataEngine(main_engine, event_engine)
    
    try:
        # 订阅比特币分钟数据
        binance_engine.subscribe("BTCUSDT", "1m")
        
        # 等待一段时间以接收数据
        print("正在接收数据，按Ctrl+C退出...")
        while True:
            time.sleep(1)
            
    except KeyboardInterrupt:
        print("\n用户中断，正在退出...")
    finally:
        # 取消订阅并停止引擎
        binance_engine.unsubscribe("BTCUSDT")
        binance_engine.stop()
        event_engine.stop()
        print("测试结束")


if __name__ == "__main__":
    main() 

开始测试Binance数据订阅...
事件引擎已启动
Binance客户端初始化成功
成功订阅BTCUSDT的1mK线数据
正在接收数据，按Ctrl+C退出...
